# Exercise 4.2: Weather Features from Open-Meteo for Frankfurt Bike-Sharing Stations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/04_spatial_data_analysis/notebooks/exercise_4_2_openmeteo_bikesharing_frankfurt.ipynb)

This feature-acquisition exercise reads a small committed Frankfurt bike-sharing station asset, fetches hourly historical weather features from Open-Meteo for every station coordinate, and visualizes the station-level context on a time-slider map.

Open-Meteo returns hourly gridded weather. Here each station receives weather features for the same 20 hourly timestamps, so the map keeps all station markers visible at every slider step.

In [ ]:
!pip -q install pandas folium requests branca

In [ ]:
from pathlib import Path
import json
import time

import branca.colormap as cm
import folium
import pandas as pd
import requests
from folium.plugins import TimestampedGeoJson

pd.set_option("display.max_columns", 100)

## 1. Load Committed Frankfurt Station Asset

The station list is small enough to keep in GitHub. The full bike-sharing archive and station-status time series stay outside GitHub, but this exercise only needs station coordinates for weather feature acquisition.

In [ ]:
STATION_ASSET_URL = "https://raw.githubusercontent.com/yfeng-hsm/KI_Geodatenanalyse_SS26/main/lectures/04_spatial_data_analysis/assets/frankfurt_bike_sharing_stations.csv"
STATION_ASSET_REPO_PATH = Path("lectures/04_spatial_data_analysis/assets/frankfurt_bike_sharing_stations.csv")
STATION_ASSET_NOTEBOOK_PATH = Path("../assets/frankfurt_bike_sharing_stations.csv")

station_asset_path = next(
    (path for path in [STATION_ASSET_REPO_PATH, STATION_ASSET_NOTEBOOK_PATH] if path.exists()),
    None,
)
station_source = station_asset_path if station_asset_path is not None else STATION_ASSET_URL

stations = pd.read_csv(station_source, dtype={"station_id": str})
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
stations = stations.dropna(subset=["lat", "lon"]).sort_values("station_id").reset_index(drop=True)

print("Station source:", station_source)
print(f"Stations: {len(stations):,}")
display(stations.head())

## 2. Define a Short Weather Time Series

The exercise uses 20 hourly timestamps on one historical date. This keeps Open-Meteo requests small while still producing a real station-by-time feature table.

In [ ]:
START_HOUR = "2022-08-26 00:00"
N_HOURS = 20
TIMEZONE = "Europe/Berlin"

start_hour = pd.Timestamp(START_HOUR, tz=TIMEZONE)
frame_hours = pd.date_range(start_hour, periods=N_HOURS, freq="1h")
frame_dates = pd.Series(frame_hours.strftime("%Y-%m-%d")).drop_duplicates().tolist()

print("Weather hours:", frame_hours.min(), "to", frame_hours.max())
print(f"Time steps: {len(frame_hours):,}")
print(f"Station-hour feature rows expected: {len(stations) * len(frame_hours):,}")
print(f"Weather dates needed: {len(frame_dates):,}")

## 3. Fetch Open-Meteo Weather Features

Open-Meteo's historical archive API supports multiple coordinates in one request. Requests are batched by station coordinate and cached locally for the current Colab/session.

In [ ]:
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FEATURES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
]
BATCH_SIZE = 40
REQUEST_SLEEP_SECONDS = 0.2
CACHE_DIR = Path("/content/openmeteo_frankfurt_cache") if Path("/content").exists() else Path("openmeteo_frankfurt_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def chunked(frame, size):
    for start in range(0, len(frame), size):
        yield frame.iloc[start : start + size].copy()


def cache_path_for(date_text, station_ids):
    return CACHE_DIR / f"openmeteo_{date_text}_{len(station_ids)}_{station_ids[0]}_{station_ids[-1]}.json"


def fetch_openmeteo_batch(date_text, station_batch):
    station_ids = station_batch["station_id"].tolist()
    path = cache_path_for(date_text, station_ids)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    params = {
        "latitude": ",".join(station_batch["lat"].map(lambda value: f"{value:.6f}")),
        "longitude": ",".join(station_batch["lon"].map(lambda value: f"{value:.6f}")),
        "start_date": date_text,
        "end_date": date_text,
        "hourly": ",".join(WEATHER_FEATURES),
        "timezone": TIMEZONE,
        "wind_speed_unit": "ms",
        "precipitation_unit": "mm",
    }
    response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    path.write_text(json.dumps(payload), encoding="utf-8")
    time.sleep(REQUEST_SLEEP_SECONDS)
    return payload


def normalize_openmeteo_payload(payload):
    return payload if isinstance(payload, list) else [payload]


def weather_rows_from_payload(date_text, station_batch, payload):
    rows = []
    for station_row, location_payload in zip(station_batch.to_dict("records"), normalize_openmeteo_payload(payload)):
        hourly = location_payload.get("hourly", {})
        for index, time_text in enumerate(hourly.get("time", [])):
            row = {
                "station_id": station_row["station_id"],
                "weather_hour": pd.Timestamp(time_text, tz=TIMEZONE),
                "weather_date": date_text,
                "openmeteo_latitude": location_payload.get("latitude"),
                "openmeteo_longitude": location_payload.get("longitude"),
            }
            for feature in WEATHER_FEATURES:
                values = hourly.get(feature, [])
                row[feature] = values[index] if index < len(values) else pd.NA
            rows.append(row)
    return rows


weather_rows = []
request_batches = 0
for date_text in frame_dates:
    for station_batch in chunked(stations, BATCH_SIZE):
        request_batches += 1
        payload = fetch_openmeteo_batch(date_text, station_batch)
        weather_rows.extend(weather_rows_from_payload(date_text, station_batch, payload))

weather = pd.DataFrame(weather_rows)
weather = weather[weather["weather_hour"].isin(frame_hours)].copy()

station_weather = stations.merge(weather, on="station_id", how="inner")
station_weather = station_weather.sort_values(["weather_hour", "station_id"]).reset_index(drop=True)

print(f"Open-Meteo request batches: {request_batches:,}")
print(f"Station-weather rows: {len(station_weather):,}")
display(station_weather.head())

## 4. Single-Time Station Map

This map fixes one timestamp and shows whether the selected weather feature differs across Frankfurt station coordinates. Marker size is fixed; only color changes.

In [ ]:
SNAPSHOT_HOUR = frame_hours[8]
SNAPSHOT_FEATURE = "temperature_2m"  # Try: precipitation, rain, relative_humidity_2m, cloud_cover, wind_speed_10m

snapshot_data = station_weather[station_weather["weather_hour"] == SNAPSHOT_HOUR].copy()
snapshot_data[SNAPSHOT_FEATURE] = pd.to_numeric(snapshot_data[SNAPSHOT_FEATURE], errors="coerce")
snapshot_data = snapshot_data.dropna(subset=[SNAPSHOT_FEATURE])

value_min = float(snapshot_data[SNAPSHOT_FEATURE].min())
value_max = float(snapshot_data[SNAPSHOT_FEATURE].max())
if value_min == value_max:
    value_max = value_min + 1

snapshot_colormap = cm.LinearColormap(
    colors=["#2c7bb6", "#ffffbf", "#d7191c"],
    vmin=value_min,
    vmax=value_max,
    caption=f"{SNAPSHOT_FEATURE} at {SNAPSHOT_HOUR}",
)

snapshot_map = folium.Map(
    location=[snapshot_data["lat"].mean(), snapshot_data["lon"].mean()],
    zoom_start=12,
    tiles="cartodbpositron",
)
for _, row in snapshot_data.iterrows():
    value = row[SNAPSHOT_FEATURE]
    popup = (
        f"<b>{row['name']}</b><br>"
        f"Station id: {row['station_id']}<br>"
        f"Hour: {row['weather_hour']}<br>"
        f"{SNAPSHOT_FEATURE}: {value}<br>"
        f"Temperature: {row['temperature_2m']} deg C<br>"
        f"Precipitation: {row['precipitation']} mm<br>"
        f"Humidity: {row['relative_humidity_2m']} %<br>"
        f"Cloud cover: {row['cloud_cover']} %<br>"
        f"Wind speed: {row['wind_speed_10m']} m/s"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="#333333",
        weight=1,
        fill=True,
        fill_color=snapshot_colormap(value),
        fill_opacity=0.82,
        popup=popup,
        tooltip=f"{row['name']}: {SNAPSHOT_FEATURE}={value}",
    ).add_to(snapshot_map)

snapshot_colormap.add_to(snapshot_map)
print(f"Snapshot rows: {len(snapshot_data):,}")
print(f"{SNAPSHOT_FEATURE} range:", snapshot_data[SNAPSHOT_FEATURE].min(), "to", snapshot_data[SNAPSHOT_FEATURE].max())
snapshot_map

## 5. Time-Series Slider Map

This second map shows every Frankfurt station at every hourly slider step. Marker size is fixed; color represents temperature. Rain, precipitation, humidity, cloud cover, and wind speed are shown in the popup.

In [ ]:
map_data = station_weather.copy()
for column in WEATHER_FEATURES:
    map_data[column] = pd.to_numeric(map_data[column], errors="coerce")
map_data = map_data.dropna(subset=["temperature_2m"]).copy()
map_data["precipitation"] = map_data["precipitation"].fillna(0)
map_data["rain"] = map_data["rain"].fillna(0)

temp_colormap = cm.LinearColormap(
    colors=["#2c7bb6", "#ffffbf", "#d7191c"],
    vmin=float(map_data["temperature_2m"].min()),
    vmax=float(map_data["temperature_2m"].max()),
    caption="Temperature at station coordinate (deg C)",
)

features = []
for _, row in map_data.iterrows():
    temperature = row["temperature_2m"]
    precipitation = row["precipitation"]
    popup = (
        f"<b>{row['name']}</b><br>"
        f"Station id: {row['station_id']}<br>"
        f"Map hour: {row['weather_hour']}<br>"
        f"Temperature: {temperature:.1f} deg C<br>"
        f"Precipitation: {precipitation:.2f} mm<br>"
        f"Rain: {row['rain']:.2f} mm<br>"
        f"Humidity: {row['relative_humidity_2m']} %<br>"
        f"Cloud cover: {row['cloud_cover']} %<br>"
        f"Wind speed: {row['wind_speed_10m']} m/s"
    )
    features.append(
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
            "properties": {
                "time": row["weather_hour"].isoformat(),
                "popup": popup,
                "tooltip": f"{row['name']}: {temperature:.1f} deg C, {precipitation:.2f} mm",
                "icon": "circle",
                "iconstyle": {
                    "fillColor": temp_colormap(temperature),
                    "fillOpacity": 0.82,
                    "stroke": "true",
                    "color": "#333333",
                    "weight": 1,
                    "radius": 6,
                },
                "style": {"color": "#333333"},
            },
        }
    )

weather_map = folium.Map(
    location=[map_data["lat"].mean(), map_data["lon"].mean()],
    zoom_start=12,
    tiles="cartodbpositron",
)
TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="PT1H",
    duration="PT1H",
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=2,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm",
    time_slider_drag_update=True,
).add_to(weather_map)
temp_colormap.add_to(weather_map)
weather_map

## 6. Questions

1. Does temperature or rain vary much across Frankfurt station coordinates, or mainly over time?
2. Why can nearby stations receive identical or very similar gridded weather values?
3. How could you reduce API calls further if you had thousands of stations?
4. Which additional station-level features would you combine with weather for a learning task?